In [4]:
import numpy as np
import matplotlib.pyplot as plt
import random
from models.qlearning import QLearningAgent

In [5]:
grid_h = 4
grid_w = 12

gridworld = np.zeros((4,12))

for h in range(grid_h):
    for w in range(grid_w):
        if(h == 3) and (w == 0 or w ==11):
            gridworld[h][w] = 0
        elif h != 3 and (w != 0 or w != 11):
            gridworld[h][w] = -1
        else:
            gridworld[h][w] = -100

terminal_state=[3,11]
initial_state = [3,0]

a_up, a_down, a_left, a_right = 0,1,2,3
actions = [a_up, a_down, a_left, a_right]

print(gridworld)

epsilon = 0.1
alpha = 0.5

gamma = 1 #for q_learning and expected_sarsa


action_symbols = [ '⬆', '⬇' , '⬅','⮕']

[[  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [   0. -100. -100. -100. -100. -100. -100. -100. -100. -100. -100.    0.]]


In [6]:
def step(state, action):
    horizontol = None
    vertical = None
    if action == 0:
        horizontol = 0
        vertical = -1
    
    elif action == 1:
        horizontol = 0
        vertical = 1

    elif action == 2:
        horizontol = -1
        vertical = 0
    else:
        horizontol = 1
        vertical = 0

    next_state = [state[0] + vertical, state[1] + horizontol]

    if next_state[0] < 0:
        next_state[0] = 0
        reward = -100
        return (next_state, reward)
    elif next_state[0] > 3:
        next_state[0] = 3
        reward = -100
        return (next_state, reward)

    if next_state[1] < 0:
        next_state[1] = 0
        reward = -100
        return (next_state, reward)
    elif next_state[1] > 11:
        next_state[1] = 11
        reward = -100
        return (next_state, reward)

    reward = gridworld[next_state[0]][next_state[1]]

    return (next_state, reward)

def episode(init_start, terminal_state, epsilon, gamma, alpha, q_values):
    time = 0
    state = init_start
    end = terminal_state
    epsilon_var = epsilon
    action = None

    explore = np.random.rand()
    if(explore <= epsilon_var):
        action = np.random.choice(actions)
    else:
        max_indexes = np.where(q_values[state[0]][state[1]] == np.max(q_values[state[0]][state[1]]))[0]
        action_idx = np.random.choice(max_indexes)
        action = actions[action_idx]


    while state != end:

        next_state, reward = step(state, action)

        explore_next = np.random.rand()
        if(explore_next <= epsilon_var):
            action_next = np.random.choice(actions)
        else:
            max_indexes = max_indexes = np.where(q_values[next_state[0]][next_state[1]] == np.max(q_values[next_state[0]][next_state[1]]))[0]
            action_idx = np.random.choice(max_indexes)
            action_next = actions[action_idx]


        error = reward + \
            (gamma*q_values[next_state[0]][next_state[1]][action_next]) \
                  - q_values[state[0]][state[1]][action]
        update = q_values[state[0]][state[1]][action] + alpha * error

        q_values[state[0]][state[1]][action] = update

        state = next_state
        action = action_next

        time += 1

        epsilon_var = epsilon_var * time

    return q_values

In [77]:
def play(episodes, q_values):

   

    for _ in range(episodes):

        q_values = episode(initial_state, terminal_state, 0.2, 1, 0.5, q_values)

        

    return q_values

    
    

In [78]:
q_values_runs = np.zeros((4,12,4))

for run in range(50):
    q_values = np.zeros((4,12,4))

    q_values_runs += play(500, q_values)

q_values_runs = q_values_runs/50



In [79]:
def policy_from_value(step, grid_h, grid_w, action_space, q_values, action_symbols):
    """
    Extract a policy (with ties) from an optimal value function.

    Args:
        step            : transition function step(state, action)
        grid_dim        : grid size (int)
        action_space    : list of actions
        optimal_values  : (grid_dim, grid_dim) numpy array V*
        action_symbols  : dict mapping action index -> printable symbol

    Returns:
        policy_str : formatted string of greedy actions per state
                    (ties allowed, shown side-by-side)
    """

    policy_str = ""
    for row in range(grid_h):
        for col in range(grid_w):
            
            best = -9999999
            best_symbols = []
            for action in action_space:
                
                temp = q_values[row][col][action]
                if(temp > best):
                    best = temp
                    action_index = action_space.index(action)
                    best_symbols = [action_index]
                elif (temp == best):
                    action_index = action_space.index(action)
                    best_symbols.append(action_index)

            text = "".join(action_symbols[ba] for ba in best_symbols)
            
            policy_str += text + "\t"
        policy_str += "\n"
    return policy_str

policy_str = policy_from_value(step, grid_h,grid_w, actions, q_values_runs, action_symbols)
print(policy_str)

⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⬇	
⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⬇	
⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⬇	
⬆	⬆	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⬆⬇⬅⮕	



In [80]:
print(gridworld)

print(q_values[3][0])
print(q_values[3,10])

[[  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [  -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.   -1.]
 [   0. -100. -100. -100. -100. -100. -100. -100. -100. -100. -100.    0.]]
[-9111.38408464 -9179.16822742 -9272.33704413 -9217.63462735]
[-2709.77331185 -3598.25996259 -3811.74351523     0.        ]


In [129]:
class SARSA:
    def __init__(self, grid_w = 12, grid_h = 4 ,start = [3,0], end = [3,11], epsilon = 0.1, gamma = 1, alpha = 0.5):
        self.start = start
        self.end = end
        self.epsilon = epsilon
        self.gamma = gamma
        self.alpha = alpha

        self.q_values_runs = np.zeros((4,12,4)) 

        self.actions = [[-1, 0], [1,0], [0, -1], [0, 1]]


        self.grid_w = grid_w
        self.grid_h = grid_h

        self.gridworld = np.zeros((grid_h,grid_w))

        for h in range(self.grid_h):
            for w in range(self. grid_w):
                if(h == 3) and (w == 0 or w ==11):
                    self.gridworld[h][w] = 0
                elif h != 3 and (w != 0 or w != 11):
                    self.gridworld[h][w] = -1
                else:
                    self.gridworld[h][w] = -100




    def __epsilon_greedy_choice(self, state, epsilon_var, q_values):
        explore = np.random.uniform(0,1)
        if(explore <= epsilon_var):
            action_index = np.random.choice([0 ,1 ,2 ,3])
            action = self.actions[action_index]
        else:
            max_indexes = np.where(q_values[state[0]][state[1]] == np.max(q_values[state[0]][state[1]]))[0]
    
            action_idx = np.random.choice(max_indexes)
            action = self.actions[action_idx]
        return action


    def __out_of_bounds(self, state):
        
        if state[0] < 0 or state[0] > 3 or state[1] < 0 or state[1] > 11:
            return True
        return False  
        

    def __step(self, state, action):
        
        vertical, horizontal = action
       
        if self.__out_of_bounds([state[0] + vertical, state[1] + horizontal]):
            reward = -1
            return state, reward

        next_state = [state[0] + vertical, state[1] + horizontal]
        reward = self.gridworld[next_state[0]][next_state[1]]

        return (next_state, reward)

    def __episode(self, q_values):
        time = 0
        state = self.start
        end = self.end
        epsilon_var = self.epsilon
        gamma = self.gamma
        alpha = self.alpha
        action = self.__epsilon_greedy_choice(state=state, epsilon_var=epsilon_var, q_values=q_values)
        
        


        while state != end:

            next_state, reward = self.__step(state, action)


            action_next = self.__epsilon_greedy_choice(state=next_state, epsilon_var=epsilon_var, q_values=q_values)
            
            error = reward + \
                (gamma*q_values[next_state[0]][next_state[1]][self.actions.index(action_next)]) \
                      - q_values[state[0]][state[1]][self.actions.index(action)]
            
            update = q_values[state[0]][state[1]][self.actions.index(action)] + alpha * error

            q_values[state[0]][state[1]][self.actions.index(action)] = update

            state = next_state
            action = action_next

            time += 1

            epsilon_var = epsilon_var * time
            
            

        return q_values
            

    def play(self, runs, episodes):

        for run in range(runs):
            q_values = np.zeros((4,12,4))

            for _ in range(episodes):

                self.q_values_runs += self.__episode(q_values)

        self.q_values_runs = self.q_values_runs/runs


        return self.q_values_runs


In [130]:

action_symbols = [ '⬆', '⬇' , '⬅','⮕']

sarsa_algo = SARSA()

q_values_runs = sarsa_algo.play(50, 500)

policy_str = policy_from_value(step, grid_h,grid_w, actions, q_values_runs, action_symbols)
print(policy_str)

⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⬇	
⬆	⬆	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⬇	
⬆	⬆	⬆	⬆	⮕	⮕	⮕	⮕	⮕	⮕	⮕	⬇	
⬆	⬆	⬆	⬆	⬆	⮕	⮕	⮕	⮕	⮕	⮕	⬆⬇⬅⮕	



## Q-Learning Agent

In [7]:
# Function to run an epsilon-greedy Q-Learning Agent on the gridworld environment

def run_ql_agent(episodes):
    
    QLAgent = QLearningAgent()
    QLAgent.initial_state = initial_state
    QLAgent.terminal = terminal_state
    QLAgent.actions = actions
    QLAgent.q_values = np.zeros((grid_h, grid_w, len(actions)))

    for _ in range(episodes):
        QLAgent.reset()
        action = QLAgent.choose_action()

        while QLAgent.state != QLAgent.terminal:
            next_state, reward = step(QLAgent.state, action)
            old_state = QLAgent.state
            QLAgent.state = next_state
            next_action = QLAgent.choose_action()

            best_action_value = np.max(QLAgent.q_values[QLAgent.state[0]][QLAgent.state[1]])
            error = reward + QLAgent.gamma * best_action_value - QLAgent.q_values[old_state[0]][old_state[1]][action]
            update = QLAgent.q_values[old_state[0]][old_state[1]][action] + QLAgent.alpha * error

            # Q-Learning Update Rule
            QLAgent.q_values[old_state[0]][old_state[1]][action] = update

            action = next_action

    return QLAgent.q_values

In [8]:
# Performs multiple runs for the Q-Learning Agent
def ql_agent_multiple_runs(runs, episodes):

    q_values_runs = np.zeros((grid_h, grid_w, len(actions)))
    
    for run in range(runs):
        q_values_runs += run_ql_agent(episodes)

    averaged_runs = q_values_runs / runs
    return averaged_runs

In [9]:
# Testing
print("One Run")
ql_agent_run_vals = run_ql_agent(150)
print(ql_agent_run_vals[2][11])
print(ql_agent_run_vals[0][10])

print("Multiple Runs")
ql_agent_values = ql_agent_multiple_runs(100, 150)
print(ql_agent_values[3][0])
print(ql_agent_values[2][5])

One Run
[ -1.97625658   0.          -1.97656249 -93.75      ]
[-89.2421875   -2.84586237  -3.25        -2.88761902]
Multiple Runs
[ -12.         -106.84628595 -107.9688404  -102.41747675]
[ -7.45221032 -96.43464245  -7.59825519  -6.        ]
